## Project Introduction

This project builds a pricing prototype for a natural gas storage contract. Using historical monthly natural gas prices, we first fit a deterministic time-series model with a linear trend and annual seasonality represented by first-order Fourier terms. The fitted model is then used to estimate the gas price for any requested date.

Using these estimated prices, we implement a contract valuation function that computes the contract value as total sale proceeds minus purchase costs and all applicable storage-related costs. The goal is to provide a clear, testable prototype that can be extended and validated before production use.

In [1]:
import pandas as pd
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.linear_model import LinearRegression

In [2]:
NatGas = pd.read_csv('Nat_Gas.csv', parse_dates= ['Dates']).set_index('Dates').to_period("M")

C:\Users\samid\AppData\Local\Temp\ipykernel_16492\528359169.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  NatGas = pd.read_csv('Nat_Gas.csv', parse_dates= ['Dates']).set_index('Dates').to_period("M")


In [3]:
fourier = CalendarFourier(freq="A", order=1)   

dp = DeterministicProcess(
    index=NatGas.index,
    constant=True,              
    order=1,                    
    additional_terms=[fourier],  
    drop=True
)

X = dp.in_sample() 

c:\Users\samid\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\deterministic.py:569: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  index = pd.date_range("2020-01-01", freq=freq, periods=1)


In [4]:
y = NatGas["Prices"]

model = LinearRegression(fit_intercept=False)
_ = model.fit(X, y)

The Fourier-based seasonal model captures the smooth annual oscillation and extends the pattern forward to generate future monthly forecasts.

In [5]:
def estimate_price(date):
    date = pd.Timestamp(date)
    
    last_period = NatGas.index[-1]      
    target_period = date.to_period("M") 
    
    steps = (target_period - last_period).n
    
    if steps <= 0:
        return float(NatGas.loc[target_period])
    
    X_fore = dp.out_of_sample(steps=steps)
    y_fore = model.predict(X_fore)
    
    return float(y_fore[-1])

In [6]:
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    injection_withdrawal_rate,
    injection_withdrawal_cost,
    max_volume,
    storage_cost_per_month
):
    
    total_profit = 0
    
    for i in range(len(injection_dates)):
        
        inj_date = pd.Timestamp(injection_dates[i])
        wdr_date = pd.Timestamp(withdrawal_dates[i])
        
        buy_price = estimate_price(inj_date)
        sell_price = estimate_price(wdr_date)

        # months between injection and withdrawal
        months = (wdr_date.year - inj_date.year) * 12 + (wdr_date.month - inj_date.month)

        # volume injected
        volume = min(injection_withdrawal_rate * months, max_volume)

        # Revenue from sale
        revenue = volume * sell_price   

        # All costs 
        cost = volume * buy_price 
        storage_cost = storage_cost_per_month * months
        fee = injection_withdrawal_cost * (volume / 1000000)

        # Total profit
        profit = revenue - cost - storage_cost - fee
        total_profit += profit
    
    return total_profit

In [ ]:
value = price_storage_contract(
    injection_dates=["2024-05-01"],
    withdrawal_dates=["2025-01-01"],
    injection_withdrawal_rate=1000000,
    injection_withdrawal_cost=10000,
    max_volume=2000000,
    storage_cost_per_month=100000
)

print("Contract Value:", value)

C:\Users\samid\AppData\Local\Temp\ipykernel_16492\1500633348.py:10: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(NatGas.loc[target_period])


Contract Value: 2685237.691309314


## Summary

A deterministic monthly pricing model was created using a linear trend and first-order Fourier seasonality to capture the dominant annual cycle in natural gas prices. A helper function was implemented to estimate the price for any requested date by mapping the date to its corresponding month and forecasting out-of-sample when needed.

A prototype storage contract pricer was then built to value a trade strategy defined by injection and withdrawal dates. The contract value is calculated as the revenue from selling stored gas minus the cost of purchasing it, minus fixed monthly storage costs and injection/withdrawal fees. This provides a simple but extensible framework for contract pricing with manual oversight.